In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder\
.appName("DataFrame")\
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/04 17:34:24 INFO SparkEnv: Registering MapOutputTracker
26/06/04 17:34:24 INFO SparkEnv: Registering BlockManagerMaster
26/06/04 17:34:24 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/06/04 17:34:24 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
data = [
    (0, "Customer_0", "Bangalore", "Karnataka", "India", "2023-11-11", True),
    (1, "Customer_1", "Delhi", "Delhi", "India", "2023-08-26", True),
]

columns = [
    "customer_id",
    "name",
    "city",
    "state",
    "country",
    "registration_date",
    "is_active"
]

In [4]:
df = spark.createDataFrame(data,columns)

In [5]:
df

DataFrame[customer_id: bigint, name: string, city: string, state: string, country: string, registration_date: string, is_active: boolean]

In [6]:
df.show()

+-----------+----------+---------+---------+-------+-----------------+---------+
|customer_id|      name|     city|    state|country|registration_date|is_active|
+-----------+----------+---------+---------+-------+-----------------+---------+
|          0|Customer_0|Bangalore|Karnataka|  India|       2023-11-11|     true|
|          1|Customer_1|    Delhi|    Delhi|  India|       2023-08-26|     true|
+-----------+----------+---------+---------+-------+-----------------+---------+



In [7]:
df.select("name").show()

+----------+
|      name|
+----------+
|Customer_0|
|Customer_1|
+----------+



### DATAFRAME --- Reading data from csv on HDFS

In [8]:
customer_df = spark.read \
.format("csv")\
.option("header","true")\
.option("inferSchema","true")\
.load("/data/customers.csv")

In [9]:
customer_df.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          0| Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|
|          2| Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|
|          3| Customer_3|Bangalore|Maharashtra|  India|       2023-03-28|    false|
|          4| Customer_4|Ahmedabad|  Karnataka|  India|       2023-06-04|    false|
|          5| Customer_5|Hyderabad| Tamil Nadu|  India|       2023-06-09|     true|
|          6| Customer_6|     Pune|  Telangana|  India|       2023-03-05|     true|
|          7| Customer_7|Ahmedabad|  Karnataka|  India|       2023-08-11|    false|
|          8| Customer_8|     Pune|  Karnataka|  India|       2023-12-28|   

In [10]:
customer_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [12]:
active_customers = customer_df.filter("is_active=true")

In [13]:
active_customers.show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|          0| Customer_0|     Pune|Maharashtra|  India|       2023-05-17|     true|
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-11-29|     true|
|          2| Customer_2|Hyderabad|Maharashtra|  India|       2023-02-08|     true|
|          5| Customer_5|Hyderabad| Tamil Nadu|  India|       2023-06-09|     true|
|          6| Customer_6|     Pune|  Telangana|  India|       2023-03-05|     true|
|         11|Customer_11|    Delhi|  Karnataka|  India|       2023-07-23|     true|
|         12|Customer_12|  Chennai|West Bengal|  India|       2023-03-08|     true|
|         14|Customer_14|Hyderabad|  Telangana|  India|       2023-02-28|     true|
|         15|Customer_15|   Mumbai|  Telangana|  India|       2023-03-17|   

In [16]:
customer_df.select("customer_id","name","city").show()

+-----------+-----------+---------+
|customer_id|       name|     city|
+-----------+-----------+---------+
|          0| Customer_0|     Pune|
|          1| Customer_1|Bangalore|
|          2| Customer_2|Hyderabad|
|          3| Customer_3|Bangalore|
|          4| Customer_4|Ahmedabad|
|          5| Customer_5|Hyderabad|
|          6| Customer_6|     Pune|
|          7| Customer_7|Ahmedabad|
|          8| Customer_8|     Pune|
|          9| Customer_9|   Mumbai|
|         10|Customer_10|     Pune|
|         11|Customer_11|    Delhi|
|         12|Customer_12|  Chennai|
|         13|Customer_13|  Chennai|
|         14|Customer_14|Hyderabad|
|         15|Customer_15|   Mumbai|
|         16|Customer_16|  Chennai|
|         17|Customer_17|Hyderabad|
|         18|Customer_18|     Pune|
|         19|Customer_19|  Kolkata|
+-----------+-----------+---------+
only showing top 20 rows



### TRANSFORMATION VS ACTION

In [20]:
#transformations
df3 = spark.read \
.format("csv")\
.option("header","true")\
.option("inferSchema","true")\
.load("/data/customers_500.csv")

In [21]:
from pyspark.sql.types import *

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("is_active", BooleanType(), True),
])

In [22]:
df4= spark.read \
.format("csv")\
.option("header","true")\
.schema(schema)\
.load("/data/customers_500.csv")

In [23]:
df4.count() #action

410812